# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [11]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_Token")
login(token=HF_TOKEN)

content_df = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)["train"].to_pandas()

print("Dataset loaded successfully!")
print("Rows:", len(content_df))
print("Columns:", len(content_df.columns))

Dataset loaded successfully!
Rows: 519606
Columns: 26


## 1. Question

### Research question

Which content items show the strongest observable signals that they may be good candidates for refresh, and how can those signals be used to prioritize human review?

### Decision supported

This analysis is designed to support a practical content-refresh prioritization decision: which pages should a reviewer investigate first.

The goal is not to claim that the model can guarantee future traffic or prove that refreshing a page will cause better performance. Instead, the analysis provides directional, decision-support signals based on observed content and search-performance data.


In [12]:
# Section 1: Question check

research_question = (
    "Which content items show the strongest observable signals "
    "that they may be good candidates for refresh?"
)

decision_supported = (
    "Prioritize content items for human review."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Which content items show the strongest observable signals that they may be good candidates for refresh?

Decision supported:
Prioritize content items for human review.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

This analysis uses the FlyRank internship warehouse release provided for the ML track.

The main content table contains content-level information for content items and client identifiers used for grouping. The query-performance data contains 90-day search performance and recent-period comparison fields.

The warehouse covers historical data through June 30, 2026. The final June 2026 period is treated as a sealed outcome window and was not used to develop label logic.

Client identifiers are used only for grouping and validation, not as model features. Private client names, URLs, and search queries are excluded from the public analysis.

The analysis also avoids target-derived fields such as `trend_direction` and `trend_pct` when they are related to the decline label.


In [13]:
print("Content rows:", len(content_df))
print("Content columns:", len(content_df.columns))

print("\nContent date-related columns:")
for col in content_df.columns:
    if "date" in col.lower():
        print("-", col)

print("\nClient column available:",
      "client_hash_id" in content_df.columns)

Content rows: 519606
Content columns: 26

Content date-related columns:
- content_created_date
- content_updated_date
- keyword_created_date
- last_optimized_date
- optimization_eligible_date

Client column available: True


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

The analysis treats the task as a content-prioritization problem rather than an automated content-generation problem.

The baseline is a simple rule-based score using content staleness and search volume. Content with a long time since optimization receives a higher priority, with an additional signal for relatively high search volume.

The model explored in the ML work is Logistic Regression because it is interpretable and provides a probability that can be used for ranking.

Client identifiers are used for grouped train/test splitting so that content from the same client does not appear in both sets.

The validation audit checks whether the model uses information that would not have been available at prediction time. Target-derived fields and overlapping outcome-period fields are excluded from the final feature set.

The results are interpreted as measured associations and directional decision-support signals, not causal evidence that a refresh will improve performance.


In [14]:
from sklearn.model_selection import GroupShuffleSplit

model_df = content_df.copy()
model_df = model_df.dropna(subset=["client_hash_id"]).copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())
print("Client overlap:", len(overlap))

Train rows: 439038
Test rows: 80568
Train clients: 67
Test clients: 17
Client overlap: 0


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results

The baseline and model results should be compared using the same evaluation split and the same metric.

The validation audit showed why split design and feature timing matter. A standard random split can allow content from the same client to appear in both training and testing, while a grouped split prevents this overlap.

Therefore, the grouped result is treated as the more conservative estimate of generalization across clients.

The model results are interpreted as predictive performance on the available evaluation target only. They should not be interpreted as proof that the recommended content actions will produce higher traffic or rankings.


In [15]:
## 4. Results

print("=== Validation Results ===")

print("Random/standard split: Week-5 comparison")
print("Grouped split: client-level 80/20 split")
print("Client overlap in grouped split:", len(overlap))

print("\nGrouped split summary:")
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

print("\nValidation conclusion:")
if len(overlap) == 0:
    print("✓ No client overlap between training and test sets.")
    print("✓ Grouped validation provides a more conservative estimate of generalization.")
else:
    print("⚠️ Client overlap detected — validation split should be checked.")

=== Validation Results ===
Random/standard split: Week-5 comparison
Grouped split: client-level 80/20 split
Client overlap in grouped split: 0

Grouped split summary:
Training rows: 439038
Test rows: 80568
Training clients: 67
Test clients: 17

Validation conclusion:
✓ No client overlap between training and test sets.
✓ Grouped validation provides a more conservative estimate of generalization.


## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

This analysis is observational and does not establish causation.

A high priority score does not prove that refreshing a page will increase traffic, clicks, rankings, or revenue.

The available data can contain differences between clients, content types, search demand, content age, and other factors that are difficult to fully control.

The model and baseline are also limited by the available historical features and outcome definitions.

Validation can reduce some sources of optimistic evaluation, but it cannot remove all confounding or guarantee performance on future data.

The results should therefore be treated as directional evidence and decision-support signals for human review, not as an autonomous content-management system.


In [16]:
limitations = [
    "Observational data",
    "No causal claim",
    "Client differences",
    "Potential confounding",
    "Historical-data dependence",
    "Human review required"
]

print("Limitations:")
for item in limitations:
    print("✓", item)

Limitations:
✓ Observational data
✓ No causal claim
✓ Client differences
✓ Potential confounding
✓ Historical-data dependence
✓ Human review required


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked recommendations

The action playbook prioritizes pages using observable signals rather than treating the score as an automatic decision.

### Recommended action hierarchy

1. **STALE_HIGH_VOLUME → REFRESH**  
   Review stale content with relatively high search demand first.

2. **STALE → REVIEW_REFRESH**  
   Review older content for declining relevance, outdated information, or opportunities to improve coverage.

3. **HIGH_VOLUME → MONITOR**  
   Monitor high-demand content even when it does not currently meet the staleness threshold.

4. **LOW_SIGNAL → NO_ACTION**  
   Do not prioritize content when the available signals provide little evidence for immediate action.

The relationship between content age, freshness, and performance should be interpreted carefully. Older content may recover when refreshed, but this analysis does not prove that refreshing every old page will improve performance.

All recommendations require human review before implementation.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## 7. Artifacts the paper embeds

The paper uses the ranked action queue and supporting summary figures generated by this notebook.

The main artifact is the ranked content-action queue. A summary chart shows how many content items fall into each recommended action category.

These artifacts are generated from the notebook so that the published paper can be traced back to the analysis.

**                                  bold 5-minute demo outline  text**

## ML-12 — 5-Minute Demo Outline

### 1. Question — 45 seconds

Which content items show the strongest observable signals that they may be good candidates for refresh, and how can those signals help prioritize human review?

This question comes from the FlyRank content-refresh problem: deciding which pages deserve attention first when there are many content items to review.

### 2. Method — 1 minute

I used a simple rule-based baseline based on content staleness and search volume, and explored Logistic Regression as an interpretable ranking model.

For validation, I used an 80/20 grouped split by client so that content from the same client does not appear in both training and test sets.

I also performed leakage checks to avoid using target-derived or outcome-period information as model features.

### 3. One Chart — 1 minute

Show the **"Content Items by Recommended Action"** chart.

Explain that the chart shows how the content queue is distributed across REFRESH, REVIEW_REFRESH, MONITOR, and NO_ACTION categories.

### 4. One Honest Result — 1 minute

The analysis shows that simple observable signals such as content staleness and search demand can be used to create a practical prioritization queue.

The grouped validation design provides a more conservative evaluation than a standard random split because clients are kept separate.

This is directional evidence for prioritization, not proof that refreshing a page will cause better traffic or rankings.

### 5. One Recommendation — 1 minute 15 seconds

Start human review with **STALE_HIGH_VOLUME** pages because they combine long time since optimization with relatively high search demand.

Reviewers should then check search intent, current performance, content quality, freshness, and business relevance before making any change.

The model or score should support the decision — not make the decision automatically.

## ML-12 — Shareable Cuts

### Short Social Post

I built a content-refresh prioritization workflow using FlyRank's anonymized internship dataset.

Instead of treating a model score as a guaranteed SEO outcome, I focused on honest validation: a simple staleness + search-volume baseline, client-grouped evaluation, and leakage checks to make sure future or target-derived information was not used incorrectly.

The result is a practical decision-support workflow that ranks content for human review. The key lesson: in SEO, a useful model is not just about prediction — it is about using the available evidence carefully and being clear about what the data can and cannot prove.

### Employer-Facing Summary

I built a content-refresh prioritization workflow using the FlyRank internship warehouse, combining a rule-based baseline, interpretable ML methodology, grouped client-level validation, leakage checks, and a ranked action playbook. The analysis showed that observable signals such as content staleness and search demand can support practical prioritization of pages for human review, while avoiding unsupported causal claims about the effect of refreshing content. The final deliverable is a reproducible research notebook and public-safe research paper that turns the analysis into actionable recommendations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
